## Imports

In [28]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from sklearn.cluster import DBSCAN, KMeans

In [29]:
def plot_data_3d(data):
    fig = px.scatter_3d(data, x='x', y='y', z='z', color='intensity', title='Puntos LiDAR - Todos los canales (3D)', labels={'x':'Coordenada X (metros)', 'y':'Coordenada Y (metros)', 'z':'Coordenada Z (metros)', 'intensity':'Intensidad'})
    fig.update_traces(marker=dict(size=2))
    fig.show()

In [30]:
# Here we just have x, y, z
def preprocess_data(data, road):
    
    # Get relevant features and compress a bit (from float 64)
    data_points = data.astype(np.float32).values

    # Create a data structure for quick neighbour lookup
    tree_data = KDTree(data_points)

    # Calculate the indices of the K nearest neighbours for each of the road data point
    _, indices_data = tree_data.query(road,k=4)

    # Delete the resulting points
    data = data[~data.index.isin(np.unique(indices_data))]

    return data

In [31]:
def preprocess_road(road):
    # Remove origin
    road = road[road["range"] != 0].reset_index()

    # Get relevant features and compress a bit (from float 64)
    road = road[["x", "y", "z"]].astype(np.float32).values
    
    return road

In [32]:
frame = pd.read_csv('/home/eder/projects/car-cluster-project/Data/pointclouds/pointcloud_1727346186_488981170.csv', sep=',')
road = pd.read_csv('/home/eder/projects/car-cluster-project/Data/carretera.csv', sep=',')

In [33]:
fig = px.scatter_3d(frame, x='x', y='y', z='z', title='Puntos LiDAR - Todos los canales (3D)', labels={'x':'Coordenada X (metros)', 'y':'Coordenada Y (metros)', 'z':'Coordenada Z (metros)', 'intensity':'Intensidad'})
fig.update_traces(marker=dict(size=2))
fig.show()

In [34]:
road = preprocess_road(road)

In [35]:
frame = preprocess_data(frame, road)

In [ ]:
# Crea un objeto DBSCAN
dbscan = DBSCAN(eps=0.8, min_samples=1200, n_jobs=-1)

# Aplica el algoritmo
y_dbscan = dbscan.fit_predict(frame)

In [45]:
fig = px.scatter_3d(frame, x='x', y='y', z='z', color=y_dbscan, title='Puntos LiDAR - Todos los canales (3D)', labels={'x':'Coordenada X (metros)', 'y':'Coordenada Y (metros)', 'z':'Coordenada Z (metros)', 'intensity':'Intensidad'})
fig.update_traces(marker=dict(size=2))
fig.show()